# HW 2 Map/Reduce

- Create a Jupyter notebook.  The name of the notebook should be your name, so I know which is whose.
- Download the data file, placing it in the same directory as the notebook.
- Then solve each of the three following assignment.

Important: Do not clear the outputs before submitting the assignment -- keep all of the values there, so that I don't necessarily have to run your notebook, I can just see the results.  If I want to run the notebook I can.  If you do clear the outputs so that I am forced to rerun your notebook, then you lose 10% on the grade.

Submit the assignment to Canvas.

## PART 1

Write a map reduce program that will examine this file, then print, for each column, the name of the column and the largest value found in that column, and for each row, the name of the row and the smallest value found in that row.  Note that the column and row names are strings, a single character.  The output should be of the form:

"A", 564
"B", 329

There should be 20 lines in the report, and they do not have to be in any particular order.

In [137]:
%%file rowcol.py

import re
import json

from mrjob.job import MRJob

# NAME = re.compile()
# DATA Line: A,K,923
# Columns ares first entry, Rows are third entry

class rowcol(MRJob):

    def mapper(self, _, line):
        val = line.strip()
        
        (col,row,entry) = (val[0:1],val[2:3],val[4:8])
        entry = int(entry)

        yield col, {"type":"column", "entry":entry, "count":1}
            # if: yield key, {values}
        yield row, {"type":"row", "entry":entry, "count":1}
            # if: yield key, {values}

        # print(val,entry,col)

    def reducer(self, key, values):
        col0 = []
        row0 = []
      
        for value in values:
            if value["type"] == "column":
                col0.append(value["entry"])
                # yield key, value["entry"]
            else:
                row0.append(value["entry"])
                # yield key, value["entry"]

        if len(col0) > 0:
            maxcol = max(col0)
            yield key, maxcol
        if len(row0) > 0:
            minrow = min(row0)
            yield key, minrow
            
if __name__ == '__main__':
    rowcol.run()

Overwriting rowcol.py


In [138]:
!python "rowcol.py" -r inline "C:/Users/anici/Class-Projects/CS4650/cs4650hw1.dat"

"A"	994
"B"	999
"C"	987
"D"	995
"E"	998
"F"	997
"G"	997
"H"	997
"I"	995
"J"	992
"K"	0
"L"	3
"M"	0
"N"	13
"O"	1
"P"	12
"Q"	0
"R"	2
"S"	5
"T"	2


No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory C:\Users\anici\AppData\Local\Temp\rowcol.anici.20261007.051213.757923
Running step 1 of 1...
job output is in C:\Users\anici\AppData\Local\Temp\rowcol.anici.20261007.051213.757923\output
Streaming final output from C:\Users\anici\AppData\Local\Temp\rowcol.anici.20261007.051213.757923\output...
Removing temp directory C:\Users\anici\AppData\Local\Temp\rowcol.anici.20261007.051213.757923...


## Part 2

This is a second map reduce program that is similar to the first part, but has an additional feature.  For a column, the output should show the column name, the largest value in that column, and the name of one of the rows where this maximum value is found (there may be more than one cell with the same maximum value).  For each row, the output should show the row name, the smallest value in that row, and the name of one of the columns where this minimum value is found.

Again, the output should have 20 lines, and they do not have to be in any order.  An example of one line of the report is:

"A", {"value":564, "example":"M"}

In [139]:
%%file rowcol2.py

import re
import json

from mrjob.job import MRJob

# NAME = re.compile()
# DATA Line: A,K,923
# Columns ares first entry, Rows are third entry

class rowcol2(MRJob):

    def mapper(self, _, line):
        val = line.strip()
        
        (col,row,entry) = (val[0:1],val[2:3],val[4:8])
        entry = int(entry)
        row = str(row)
        col = str(col)

        yield col, {"type":"column", "sub":row, "entry":entry, "count":1}
            # if: yield key, {values}
        yield row, {"type":"row", "sub":col, "entry":entry, "count":1}
            # if: yield key, {values}

        # print(val,entry,col)

    def reducer(self, key, values):
        col0 = []
        row0 = []
        colsub = []
        rowsub = []
      
        for value in values:
            if value["type"] == "column":
                col0.append(value["entry"])
                row01 = value["sub"]
                colsub.append(row01)
                # yield key, {"value":value["entry"], "example":row01}
            else:
                row0.append(value["entry"])
                col01 = value["sub"]
                rowsub.append(col01)
                # yield key, {"value":value["entry"], "example":col01}

        if len(col0) > 0:
            maxcol = max(col0)
            idx = col0.index(maxcol)
            yield key, {"value":maxcol, "example":colsub[idx]}
        if len(row0) > 0:
            minrow = min(row0)
            idx = row0.index(minrow)
            yield key, {"value":minrow, "example":rowsub[idx]}
            
if __name__ == '__main__':
    rowcol2.run()

Overwriting rowcol2.py


In [140]:
!python "rowcol2.py" -r inline "C:/Users/anici/Class-Projects/CS4650/cs4650hw1.dat"

"A"	{"value":994,"example":"R"}
"B"	{"value":999,"example":"N"}
"C"	{"value":987,"example":"S"}
"D"	{"value":995,"example":"M"}
"E"	{"value":998,"example":"N"}
"F"	{"value":997,"example":"M"}
"G"	{"value":997,"example":"M"}
"H"	{"value":997,"example":"P"}
"I"	{"value":995,"example":"S"}
"J"	{"value":992,"example":"K"}
"K"	{"value":0,"example":"H"}
"L"	{"value":3,"example":"H"}
"M"	{"value":0,"example":"E"}
"N"	{"value":13,"example":"B"}
"O"	{"value":1,"example":"H"}
"P"	{"value":12,"example":"B"}
"Q"	{"value":0,"example":"C"}
"R"	{"value":2,"example":"F"}
"S"	{"value":5,"example":"I"}
"T"	{"value":2,"example":"G"}


No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory C:\Users\anici\AppData\Local\Temp\rowcol2.anici.20261007.051216.124387
Running step 1 of 1...
job output is in C:\Users\anici\AppData\Local\Temp\rowcol2.anici.20261007.051216.124387\output
Streaming final output from C:\Users\anici\AppData\Local\Temp\rowcol2.anici.20261007.051216.124387\output...
Removing temp directory C:\Users\anici\AppData\Local\Temp\rowcol2.anici.20261007.051216.124387...


## Part 3

This is a third map reduce program that is a further extension of Part 2.  In this case, for the output, we want to include a list of all rows/columns that have that maximum/minimum value.  Note that most rows and columns will have a unique extreme, but some will have duplicates.  This output might look like this:

"A", ["value":564, "examples": ["M"]}
"D", ["value":437, "examples": ["L",  "P"]}

In [141]:
%%file rowcol3.py

import re
import json

from mrjob.job import MRJob

# NAME = re.compile()
# DATA Line: A,K,923
# Columns ares first entry, Rows are third entry

class rowcol3(MRJob):

    def mapper(self, _, line):
        val = line.strip()
        
        (col,row,entry) = (val[0:1],val[2:3],val[4:8])
        entry = int(entry)
        row = str(row)
        col = str(col)

        yield col, {"type":"column", "sub":row, "entry":entry, "count":1}
            # if: yield key, {values}
        yield row, {"type":"row", "sub":col, "entry":entry, "count":1}
            # if: yield key, {values}

        # print(val,entry,col)

    def reducer(self, key, values):
        col0 = []   # all values
        row0 = []
        colsub = [] # all rows/cols from each value
        rowsub = []
      
        for value in values:
            if value["type"] == "column":
                col0.append(value["entry"])
                row01 = value["sub"]
                colsub.append(row01)
                # yield key, {"value":value["entry"], "example":row01}
            else:
                row0.append(value["entry"])
                col01 = value["sub"]
                rowsub.append(col01)
                # yield key, {"value":value["entry"], "example":col01}

        # function: input max or min value, output list of indices
        def maxminindex(zero,zero0):
            indices = []
            for i in range(len(zero0)):
                if zero0[i] == zero:
                    indices.append(i)
            return indices            

        # function: input list of indices --> output values from another lsit of the indices
        def listlist(ids,ids0):
            colsub0 = []
            for i in ids:
                colsub0.append(ids0[i])
            return colsub0            

        # run function within a function

        if len(col0) > 0:
            maxcol = max(col0)
            idx = listlist(maxminindex(maxcol,col0),colsub)
            yield key, {"value":maxcol, "example":idx}
        if len(row0) > 0:
            minrow = min(row0)
            idx = listlist(maxminindex(minrow,row0),rowsub)
            yield key, {"value":minrow, "example":idx}
            
if __name__ == '__main__':
    rowcol3.run()

Overwriting rowcol3.py


In [142]:
!python "rowcol3.py" -r inline "C:/Users/anici/Class-Projects/CS4650/cs4650hw1.dat"

"A"	{"value":994,"example":["R","P"]}
"B"	{"value":999,"example":["N"]}
"C"	{"value":987,"example":["S"]}
"D"	{"value":995,"example":["M"]}
"E"	{"value":998,"example":["N","S"]}
"F"	{"value":997,"example":["M"]}
"G"	{"value":997,"example":["M"]}
"H"	{"value":997,"example":["P"]}
"I"	{"value":995,"example":["S"]}
"J"	{"value":992,"example":["K"]}
"K"	{"value":0,"example":["H"]}
"L"	{"value":3,"example":["H","B"]}
"M"	{"value":0,"example":["E"]}
"N"	{"value":13,"example":["B"]}
"O"	{"value":1,"example":["H"]}
"P"	{"value":12,"example":["B"]}
"Q"	{"value":0,"example":["C","G"]}
"R"	{"value":2,"example":["F"]}
"S"	{"value":5,"example":["I"]}
"T"	{"value":2,"example":["G"]}


No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory C:\Users\anici\AppData\Local\Temp\rowcol3.anici.20261007.051219.038052
Running step 1 of 1...
job output is in C:\Users\anici\AppData\Local\Temp\rowcol3.anici.20261007.051219.038052\output
Streaming final output from C:\Users\anici\AppData\Local\Temp\rowcol3.anici.20261007.051219.038052\output...
Removing temp directory C:\Users\anici\AppData\Local\Temp\rowcol3.anici.20261007.051219.038052...
